# AML s26 - Intro to PyTorch


---

## Table of Contents
1. [What is PyTorch?](#1.-What-is-PyTorch?)
2. [Tensors](#2.-Tensors)
3. [Autograd — Automatic Differentiation](#3.-Autograd-—-Automatic-Differentiation)
4. [Building a Neural Network](#4.-Building-a-Neural-Network)
5. [The Training Loop](#5.-The-Training-Loop)
6. [GPU Acceleration](#6.-GPU-Acceleration)
7. [Saving & Loading Models](#7.-Saving-&-Loading-Models)
8. [End-to-End Example](#8.-End-to-End-Example)

## Setup

Install PyTorch if you haven't already:

>    conda install -c conda-forge pytorch

---
## Hardware acceleration
### a) macOS
- PyTorch provides GPU acceleration for macOS devices through the Metal Performance Shaders (MPS) backend. 
- This allows users to leverage the powerful integrated GPUs in Apple Silicon (M1, M2, M3, M4, M5). 
- Requirements:  macOS >= 12.3, PyTorch >= 1.12 

### b) PC with  NVIDIA GPU 
- Different installation procedure!
- Requires NVIDIA drivers and CUDA
- PyTorch, NVIDIA driver, and CUDA versions need to match!
- Search online for more information 

In [2]:

#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/xpu
import torch
import torch.nn as nn
import torch.optim as optim

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
print(f'MPS available:  {torch.backends.mps.is_available()}')
print(f'XPU available:  {torch.xpu.is_available()}')

PyTorch version: 2.11.0+xpu
CUDA available:  False
MPS available:  False
XPU available:  True


---
## 1. What is PyTorch?

PyTorch is an open-source deep learning framework built by **Meta AI** (released 2016).

Think of it as **NumPy optimized for deep learning**:
- Runs computations on **GPUs**
- Builds and trains **neural networks**
- Computes **gradients automatically** (autograd)
- Is the **#1 framework in academic research**

### PyTorch vs NumPy — Overview

| NumPy | PyTorch | Notes |
|---|---|---|
| `np.array(...)` | `torch.tensor(...)` | Core data type |
| `arr.shape` | `tensor.shape` | Identical API |
| `arr.reshape(...)` | `tensor.reshape(...)` | Identical API |
| `arr + arr` | `tensor + tensor` | Identical API |
| `arr.mean()` | `tensor.mean()` | Identical API |
| — | `tensor.backward()` | **PyTorch only** — gradients |
| — | `tensor.cuda()` | **PyTorch only** — GPU |


- If you know Python and NumPy, using PyTorch is easy.

---
## 2. Tensors

A **tensor** is an n-dimensional array — the fundamental data structure in PyTorch.
It works exactly like `np.ndarray`, but can live on a GPU and track gradients.

### 2.1 Creating Tensors

In [3]:
import torch

# From a Python list
a = torch.tensor([1.0, 2.0, 3.0])
print('1D tensor:', a)
print('shape:    ', a.shape)
print('dtype:    ', a.dtype)

# 2D tensor (matrix)
b = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
print('\n2D tensor:')
print(b)
print('shape: ', b.shape)
print('dtype: ', b.dtype)

1D tensor: tensor([1., 2., 3.])
shape:     torch.Size([3])
dtype:     torch.float32

2D tensor:
tensor([[1, 2, 3],
        [4, 5, 6]])
shape:  torch.Size([2, 3])
dtype:  torch.int64


In [4]:
# Factory functions - creates and returns a new tensor
print('zeros:  ', torch.zeros(2, 3))
print('ones:   ', torch.ones(2, 3))
print('rand:   ', torch.rand(2, 3))     # uniform [0, 1)
print('randn:  ', torch.randn(2, 3))    # standard normal
print('eye:    ', torch.eye(3))          # identity matrix
print('arange: ', torch.arange(0, 10, 2))

zeros:   tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:    tensor([[1., 1., 1.],
        [1., 1., 1.]])
rand:    tensor([[0.0084, 0.7772, 0.9715],
        [0.7385, 0.1743, 0.3052]])
randn:   tensor([[-1.7105,  1.6632, -0.1056],
        [-1.4895, -0.5967,  1.0236]])
eye:     tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
arange:  tensor([0, 2, 4, 6, 8])


### 2.2 Tensor Operations

PyTorch tensors support all standard arithmetic — element-wise and matrix operations.

In [5]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])

# Element-wise arithmetic
print('x + y  =', x + y)
print('x * y  =', x * y)
print('x ** 2 =', x ** 2)

# Reductions
print('sum:  ', x.sum())
print('mean: ', x.mean())
print('max:  ', x.max())

# Matrix multiplication (use @ operator)
A = torch.randn(3, 4)
B = torch.randn(4, 5)
C = A @ B                # equivalent to torch.matmul(A, B)
print('\nA @ B shape:', C.shape)

x + y  = tensor([5., 7., 9.])
x * y  = tensor([ 4., 10., 18.])
x ** 2 = tensor([1., 4., 9.])
sum:   tensor(6.)
mean:  tensor(2.)
max:   tensor(3.)

A @ B shape: torch.Size([3, 5])


### 2.3 Shapes and Reshaping

Understanding and manipulating shapes is crucial in deep learning.
Batch dimensions, channel dimensions, sequence lengths --> it all comes down to shapes.

In [6]:
x = torch.randn(12)   # flat vector

print('Original:      ', x.shape)
print('reshape(3,4):  ', x.reshape(3, 4).shape)
print('reshape(2,2,3):', x.reshape(2, 2, 3).shape)

a = x.reshape(3, 4)

# Add / remove dimensions
print('unsqueeze(0):  ', a.unsqueeze(0).shape)   # (1, 3, 4) — add batch dim
print('squeeze:       ', a.unsqueeze(0).squeeze(0).shape)  # back to (3, 4)

# Permute  (like np.transpose for arbitrary dims)
print('permute(1,0):  ', a.permute(1, 0).shape)  # (4, 3)

# Useful properties
print(f'\nndim:   {a.ndim}')
print(f'numel:  {a.numel()}')   # total number of elements
print(f'device: {a.device}')   # cpu or cuda
print(f'dtype:  {a.dtype}')

Original:       torch.Size([12])
reshape(3,4):   torch.Size([3, 4])
reshape(2,2,3): torch.Size([2, 2, 3])
unsqueeze(0):   torch.Size([1, 3, 4])
squeeze:        torch.Size([3, 4])
permute(1,0):   torch.Size([4, 3])

ndim:   2
numel:  12
device: cpu
dtype:  torch.float32


### 2.4 Data Types

PyTorch has a strict type system. The most common dtypes are:

| dtype | alias | when to use |
|---|---|---|
| `torch.float32` | `torch.float` | **Default** for model weights & activations |
| `torch.float64` | `torch.double` | Rarely needed in deep learning |
| `torch.int64` | `torch.long` | **Class labels**, indices |
| `torch.bool` | — | Masks |

In [7]:
# Specify dtype at creation
x = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
labels = torch.tensor([0, 1, 2], dtype=torch.long)  # int64 for class indices

print('float32:', x.dtype)
print('int64:  ', labels.dtype)

# Type conversion
print('to int: ', x.to(torch.int64))
print('to bool:', (x > 1.5))   # tensor([False,  True,  True])


float32: torch.float32
int64:   torch.int64
to int:  tensor([1, 2, 3])
to bool: tensor([False,  True,  True])


---
## 3. Autograd — Automatic Differentiation

- Training a neural network requires computing gradients 
    — derivatives of the loss
with respect to every parameter. 
- PyTorch does this **automatically**.

- If  `requires_grad=True` on a tensor, PyTorch records every operation on it into a **computation graph**. 
- Calling `.backward()` walks that graph in reverse to compute all gradients.

### 3.1 Basic Gradient Computation

In [8]:
x = torch.tensor(3.0, requires_grad=True)

# Define a computation
y = x ** 2 + 2 * x + 1   # y = x² + 2x + 1

# Compute gradients — dy/dx = 2x + 2
y.backward()

print(f'y      = {y.item():.1f}')         # 16.0  (3² + 6 + 1)
print(f'dy/dx  = {x.grad.item():.1f}')   # 8.0   (2*3 + 2)

y      = 16.0
dy/dx  = 8.0


### 3.2 Gradients with Multiple Parameters

- Define a linear model with two parameters. 
- After `.backward()`, each parameter has a `.grad` telling us which direction reduces the loss.

In [9]:
# Two learnable parameters
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)

# Input and target
x_in     = torch.tensor(1.0)
y_target = torch.tensor(5.0)

# Forward pass — linear model
y_pred = w * x_in + b           # prediction = 2.5
loss   = (y_pred - y_target)**2  # squared error

print(f'Prediction: {y_pred.item()}')
print(f'Loss:       {loss.item()}')

# Backward pass
loss.backward()

print(f'\nw.grad = {w.grad.item()}')  # d(loss)/dw
print(f'b.grad = {b.grad.item()}')  # d(loss)/db
print('\nNegative gradients → increase w and b to reduce loss')

print('\nLoss: ', loss)

Prediction: 2.5
Loss:       6.25

w.grad = -5.0
b.grad = -5.0

Negative gradients → increase w and b to reduce loss

Loss:  tensor(6.2500, grad_fn=<PowBackward0>)


### 3.3 Disabling Gradient Tracking

- During **evaluation and inference** you don't need gradients.
- Disabling them saves memory and speeds up computation.

In [10]:
x = torch.randn(3, requires_grad=True)

# Option 1: context manager (most common)
with torch.no_grad():
    y = x * 2
    print('requires_grad inside no_grad:', y.requires_grad)  # False

# Option 2: detach — share data, cut from graph
z = x.detach()
print('requires_grad after detach:  ', z.requires_grad)      # False

# Option 3: inference_mode (fastest, most restrictive)
with torch.inference_mode():
    out = x * 3
    print('requires_grad in inf mode:   ', out.requires_grad) # False

# Rule: always wrap model.eval() calls in torch.no_grad()

requires_grad inside no_grad: False
requires_grad after detach:   False
requires_grad in inf mode:    False


---
## 4. Building a Neural Network (**)

Every model in PyTorch inherits from `nn.Module`. You need to define:
- **`__init__`** — declare layers and parameters
- **`forward`** — describe the computation (forward pass)

### 4.1 Custom `nn.Module`

In [11]:
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self, input_size=784, hidden=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden)       # input -> hidden
        self.fc2 = nn.Linear(hidden, hidden // 2)      # hidden -> smaller
        self.fc3 = nn.Linear(hidden // 2, num_classes)  # -> output
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))   # activation after layer 1
        x = self.dropout(x)           # regularization
        x = torch.relu(self.fc2(x))   # activation after layer 2
        x = self.fc3(x)               # raw logits — no softmax
        return x

model = SimpleNet()
print(model)

SimpleNet(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [12]:
# Count parameters
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,}')

# Quick forward pass with fake data
x_fake = torch.randn(32, 784)  # batch of 32 'images' (flattened 28×28)
out    = model(x_fake)
print(f'\nInput shape:  {x_fake.shape}')
print(f'Output shape: {out.shape}')   # (32, 10) — one score per class

Total params:     109,386
Trainable params: 109,386

Input shape:  torch.Size([32, 784])
Output shape: torch.Size([32, 10])


### 4.2 `nn.Sequential` — Quick Stacks

For simple feed-forward architectures, `nn.Sequential` removes the boilerplate.

In [ ]:
model_seq = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 10)
)

x = torch.randn(16, 784)
out = model_seq(x)
print('Output shape:', out.shape)   # (16, 10)

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on xpu:0, different from other tensors on cpu (when checking argument in method wrapper_XPU_addmm)

### 4.3 Common Built-in Layers

In [14]:
# ── Fully connected ────────────────────────────────────────────────────
fc = nn.Linear(in_features=128, out_features=64)
print('Linear output:', fc(torch.randn(4, 128)).shape)  # (4, 64)

# ── Convolution ─────────────────────────────────────────────────────────
# in_channels=3 (RGB), out_channels=32, kernel_size=3
conv = nn.Conv2d(3, 32, kernel_size=3, padding=1)
img  = torch.randn(8, 3, 32, 32)   # batch=8, C=3, H=32, W=32
print('Conv2d output:', conv(img).shape)  # (8, 32, 32, 32)

# ── Normalization & Dropout ──────────────────────────────────────────────
bn   = nn.BatchNorm2d(32)
drop = nn.Dropout(p=0.3)

# ── Activation functions ─────────────────────────────────────────────────
activations = [nn.ReLU(), nn.GELU(), nn.Sigmoid(), nn.Tanh()]
x_act = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
for act in activations:
    print(f'{act.__class__.__name__:12s}: {act(x_act).detach().numpy().round(2)}')

Linear output: torch.Size([4, 64])
Conv2d output: torch.Size([8, 32, 32, 32])
ReLU        : [0. 0. 0. 1. 2.]
GELU        : [-0.05 -0.16  0.    0.84  1.95]
Sigmoid     : [0.12 0.27 0.5  0.73 0.88]
Tanh        : [-0.96 -0.76  0.    0.76  0.96]


---
## 5. The Training Loop

Training has three ingredients: 
- a **loss function**, 
- an **optimizer**, and 
- the **backpropagation loop**.

### 5.1 Loss Functions

In [15]:
# ── Classification ─────────────────────────────────────────────────────
# CrossEntropyLoss = softmax + Negative Log-Likelihood (NLL) in one step
# (!) Expects RAW LOGITS — do NOT apply softmax first
criterion = nn.CrossEntropyLoss()

logits = torch.randn(8, 10)           # batch=8, classes=10
labels = torch.randint(0, 10, (8,))  # int64 class indices | 1D tensor (a vector) with 8 elements | values 0,1,...,9
loss   = criterion(logits, labels)
print(f'Cross-entropy loss: {loss.item():.4f}')  

# ── Regression ──────────────────────────────────────────────────────────
mse = nn.MSELoss()
mae = nn.L1Loss()  # Mean Absolute Error

preds   = torch.tensor([1.0, 2.0, 3.0])
targets = torch.tensor([1.5, 2.5, 2.0])
print(f'MSE: {mse(preds, targets).item():.4f}')
print(f'MAE: {mae(preds, targets).item():.4f}')

Cross-entropy loss: 3.5712
MSE: 0.5000
MAE: 0.6667


### 5.2 Optimizers

An optimizer updates model weights using computed gradients.

In [16]:
model = SimpleNet()

# Stochastic Gradient Descent (SGD) — classic, sometimes needs careful tuning
sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam — great default choice
adam = optim.Adam(model.parameters(), lr=1e-3)

# AdamW — Adam + weight decay (preferred for modern models)
adamw = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# Pick one. AdamW is a solid default for most tasks.
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
print('Optimizer:', optimizer)

Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.01
)


### 5.3 The Core Training Loop

Every PyTorch training step follows the **same five-line pattern**:

```python
optimizer.zero_grad()                # 1) clear old gradients
output = model(X_batch)              # 2) forward pass
loss   = criterion(output, y_batch)  # 3) compute loss
loss.backward()                      # 4) backprop
optimizer.step()                     # 5) update weights
```

> - (!!!) **Never forget `zero_grad()`!** PyTorch *accumulates* gradients by default.
> - Skipping it means gradients from the previous step corrupt the current one.

In [19]:
# generate a simulated dataset 
torch.manual_seed(42)
X_fake = torch.randn(256, 784)
y_fake = torch.randint(0, 10, (256,))

model     = SimpleNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(1, 11):
    model.train()                          # activate training mode | enable dropout / batchnorm

    optimizer.zero_grad()                  # 1) clear gradients
    logits = model(X_fake)                 # 2) forward pass
    loss   = criterion(logits, y_fake)     # 3) loss
    loss.backward()                        # 4) backpropagation 
    optimizer.step()                       # 5) update weights

    print(f'Epoch {epoch:2d}  |  loss: {loss.item():.4f}')

Epoch  1  |  loss: 2.3054
Epoch  2  |  loss: 2.2638
Epoch  3  |  loss: 2.2219
Epoch  4  |  loss: 2.1699
Epoch  5  |  loss: 2.1276
Epoch  6  |  loss: 2.0752
Epoch  7  |  loss: 2.0294
Epoch  8  |  loss: 1.9689
Epoch  9  |  loss: 1.9173
Epoch 10  |  loss: 1.8691


### 5.4 Train vs Eval Mode

Always switch modes — it affects **Dropout** and **BatchNorm** behaviour.

In [20]:
# model.train()  ->  Dropout active, BatchNorm uses batch stats
# model.eval()   ->  Dropout off, BatchNorm uses running stats

# Demonstrate Dropout difference
drop_layer = nn.Dropout(p=0.5)
x = torch.ones(1, 10)

drop_layer.train()   # training mode
out_train = drop_layer(x)
print('Train mode (dropout active):  ', out_train)

drop_layer.eval()    # eval mode
out_eval = drop_layer(x)
print('Eval mode  (dropout off):     ', out_eval)

# Typical pattern in a training script:
# for epoch in range(epochs):
#     model.train()
#     for X_batch, y_batch in train_loader:
#         ...  # training step
#
#     model.eval()
#     with torch.no_grad():
#         for X_val, y_val in val_loader:
#             ...  # evaluation

Train mode (dropout active):   tensor([[0., 0., 2., 0., 2., 0., 0., 2., 0., 0.]])
Eval mode  (dropout off):      tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])


---
## 6. GPU Acceleration

- PyTorch makes GPU usage **explicit** — need to decide what lives where!
- Important: **model and data must be on the same device**!

In [21]:
# Check device availability
# DEVICE — MPS (M4), CUDA, or CPU fallback

device = (
    torch.device("xpu")  if torch.xpu.is_available() else
    torch.device("cuda") if torch.cuda.is_available()         else
    torch.device("cpu")
)

print(f'Using device: {device}')

# Moving tensors
x = torch.randn(3, 3)
x = x.to(device)          # move to GPU (no-op if device='cpu')
print('Tensor device:', x.device)

# Moving a model
model = SimpleNet()
model = model.to(device)
print('Model device:', next(model.parameters()).device)

Using device: xpu
Tensor device: xpu:0
Model device: xpu:0


In [22]:
# ── GPU-ready training loop template ────────────────────────────────────
# device  = ...
model     = SimpleNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# Simulated single batch
X_batch = torch.randn(32, 784).to(device)   #  move data to device
y_batch = torch.randint(0, 10, (32,)).to(device)

model.train()
optimizer.zero_grad()
loss = criterion(model(X_batch), y_batch)
loss.backward()
optimizer.step()

print(f'Loss: {loss.item():.4f}  |  ran on: {device}')
print('\nThis exact code runs unchanged on CPU or GPU — just change device!')

Loss: 2.3413  |  ran on: xpu

This exact code runs unchanged on CPU or GPU — just change device!


---
## 7. Saving & Loading Models

- Always save the `state_dict` (weights) rather than the whole model object.
- It's more portable and doesn't depend on your class structure at load time.

In [23]:
import os

model = SimpleNet()

# - Save weights ---------------------------
torch.save(model.state_dict(), './simple_net.pth')
print('Saved model weights.')

# - Load weights ---------------------------
model2 = SimpleNet()                              # re-create architecture
model2.load_state_dict(torch.load('./simple_net.pth', weights_only=True))
model2.eval()                                     # always eval() after loading
print('Loaded model weights.')

# Verify they're identical
x_test = torch.randn(1, 784)
with torch.no_grad():
    out1 = model(x_test)
    out2 = model2(x_test)
print('Outputs identical:', torch.allclose(out1, out2))

Saved model weights.
Loaded model weights.
Outputs identical: False


In [24]:
# - Full checkpoint (save optimizer & epoch too) --------------------------
epoch = 5
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
loss_val  = torch.tensor(0.4321)

checkpoint = {
    'epoch':     epoch,
    'model':     model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'loss':      loss_val.item(),
}
torch.save(checkpoint, './checkpoint.pth')
print('Checkpoint saved.')

# - Restore checkpoint ---------------------------------------------------
ckpt = torch.load('./checkpoint.pth', weights_only=False)
model3 = SimpleNet()
opt3   = optim.AdamW(model3.parameters(), lr=1e-3)

model3.load_state_dict(ckpt['model'])
opt3.load_state_dict(ckpt['optimizer'])
start_epoch = ckpt['epoch'] + 1

print(f'Resumed from epoch {ckpt["epoch"]}  |  loss was {ckpt["loss"]:.4f}')
print(f'Will continue from epoch {start_epoch}')

Checkpoint saved.
Resumed from epoch 5  |  loss was 0.4321
Will continue from epoch 6


---
## 8. End-to-End Example — Classification

- The "make_moons" dataset is a classic non-linearly separable binary classification problem.

In [26]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Generate Data ---------------------------------------------------
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=42)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

# Convert to tensors
X_train = torch.FloatTensor(X_train_np).to('xpu')
y_train = torch.LongTensor(y_train_np).to('xpu')
X_test  = torch.FloatTensor(X_test_np).to('xpu')
y_test  = torch.LongTensor(y_test_np).to('xpu')

print(f'Train: {X_train.shape}, labels: {y_train.shape}')
print(f'Test:  {X_test.shape},  labels: {y_test.shape}')
print(f'Classes: {y_train.unique().tolist()}')

Train: torch.Size([800, 2]), labels: torch.Size([800])
Test:  torch.Size([200, 2]),  labels: torch.Size([200])
Classes: [0, 1]


In [29]:
# - 2. Generate Model --------------------------------------------------------
torch.manual_seed(0)

moon_model = nn.Sequential(
    nn.Linear(2, 32),    # 2 input features (x, y coordinates)
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 2)     # 2 output classes
).to('xpu')

criterion = nn.CrossEntropyLoss().to('xpu')
optimizer = optim.Adam(moon_model.parameters(), lr=1e-2)

# - 3. Training --------------------------------------------------------------
history = []

for epoch in range(1, 101):
    moon_model.train()
    optimizer.zero_grad()
    loss = criterion(moon_model(X_train), y_train)
    loss.backward()
    optimizer.step()

    # Track every 10 epochs
    if epoch % 10 == 0:
        moon_model.eval()
        with torch.no_grad():
            val_preds   = moon_model(X_test).argmax(dim=1)
            val_acc     = (val_preds == y_test).float().mean().item()
            train_preds = moon_model(X_train).argmax(dim=1)
            train_acc   = (train_preds == y_train).float().mean().item()
        history.append({'epoch': epoch, 'loss': loss.item(),  'train_acc': train_acc, 'val_acc': val_acc})
        print(f'Epoch {epoch:3d}  |  loss: {loss.item():.4f}   train acc: {train_acc:.1%}      |     val acc: {val_acc:.1%}')

Epoch  10  |  loss: 0.3157   train acc: 85.7%      |     val acc: 84.0%
Epoch  20  |  loss: 0.2339   train acc: 90.2%      |     val acc: 90.5%
Epoch  30  |  loss: 0.1678   train acc: 93.6%      |     val acc: 94.5%
Epoch  40  |  loss: 0.1170   train acc: 96.1%      |     val acc: 98.0%
Epoch  50  |  loss: 0.0862   train acc: 97.0%      |     val acc: 98.0%
Epoch  60  |  loss: 0.0723   train acc: 97.2%      |     val acc: 99.0%
Epoch  70  |  loss: 0.0667   train acc: 97.5%      |     val acc: 99.0%
Epoch  80  |  loss: 0.0642   train acc: 97.7%      |     val acc: 99.0%
Epoch  90  |  loss: 0.0627   train acc: 97.7%      |     val acc: 99.0%
Epoch 100  |  loss: 0.0619   train acc: 97.6%      |     val acc: 99.0%


In [30]:
# - 4. Final Evaluation -----------------------------------------------------
moon_model.eval()
with torch.no_grad():
    test_logits = moon_model(X_test)
    test_probs  = torch.softmax(test_logits, dim=1)
    test_preds  = test_logits.argmax(dim=1)
    accuracy    = (test_preds == y_test).float().mean().item()

print(f'Final test accuracy: {accuracy:.1%}')
print(f'\nSample predictions (first 10):')
print('Predicted:', test_preds[:10].tolist())
print('Actual:   ', y_test[:10].tolist())

Final test accuracy: 99.0%

Sample predictions (first 10):
Predicted: [1, 0, 1, 0, 1, 1, 0, 0, 1, 0]
Actual:    [1, 0, 1, 0, 1, 1, 0, 0, 1, 0]


In [34]:
# - 5. Visualize the decision boundary -------------------------------------

import matplotlib.pyplot as plt

h = 0.02
x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                        np.arange(y_min, y_max, h))

grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]).cpu()
moon_model.eval()
with torch.no_grad():
    Z = moon_model(grid).argmax(dim=1).numpy()
Z = Z.reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision boundary
axes[0].contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
axes[0].scatter(X_test_np[:, 0], X_test_np[:, 1],
                c=y_test_np, cmap='coolwarm', edgecolors='k', s=40)
axes[0].set_title(f'Decision Boundary (test acc: {accuracy:.1%})', fontsize=13)
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')

# Training history
epochs_hist  = [h['epoch']    for h in history]
train_accs   = [h['train_acc'] for h in history]
val_accs     = [h['val_acc']   for h in history]
axes[1].plot(epochs_hist, train_accs, 'o-', label='Train', color='steelblue')
axes[1].plot(epochs_hist, val_accs,   's--', label='Val',   color='tomato')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training History', fontsize=13)
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()
print('Plots rendered.')



RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on xpu:0 (when checking argument in method wrapper_XPU_addmm)

---
## 9. More PyTorch ...



| Topic | What to Learn | Key API |
|---|---|---|
| **DataLoaders** | Batching, shuffling, custom datasets | `torch.utils.data.Dataset`, `DataLoader` |
| **CNNs** | Image classification | `nn.Conv2d`, `nn.MaxPool2d` |
| **Transfer Learning** | Pretrained models | `torchvision.models` |
| **LR Schedulers** | Adaptive learning rate | `torch.optim.lr_scheduler` |
| **RNNs / LSTMs** | Sequences & time-series | `nn.LSTM`, `nn.GRU` |
| **Transformers** | NLP / vision transformers | Hugging Face `transformers` |


---

## Summary

| Concept | One-liner |
|---|---|
| `torch.Tensor` | n-dimensional array, GPU-aware |
| `requires_grad=True` | opt into the computation graph |
| `.backward()` | compute all gradients via backprop |
| `nn.Module` | base class for every model |
| `nn.Linear` | fully connected layer: y = xWᵀ + b |
| `optimizer.zero_grad()` | **always** clear gradients before each step |
| `model.train()` | enable dropout / batchnorm training behaviour |
| `model.eval()` | disable dropout, use running stats |
| `torch.no_grad()` | skip gradient tracking during inference |
| `.to(device)` | move tensor or model to CPU / GPU |

---

## Resources

- **Official tutorials:** [pytorch.org/tutorials](https://pytorch.org/tutorials)
- **API reference:** [pytorch.org/docs](https://pytorch.org/docs/stable/index.html)
